# T1, T2, and Ramsey experiments

This notebook collects the three standard coherence measurements:
energy relaxation (T1), echo-based dephasing (T2), and Ramsey detuning.
It also includes an optional spectator-state Ramsey check when a second
qubit is available.

## 1. Create an `Experiment`

Pick at least one qubit. If you want to run the spectator-state Ramsey
cell later, choose two qubits from the same device configuration.

In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

In [ ]:
Q0 = exp.qubit_labels[0]
spectator = exp.qubit_labels[1] if len(exp.qubit_labels) > 1 else None

print("target qubit:", Q0)
print("spectator:", spectator)

## 2. Connect and prepare the basic pi pulse

A Rabi fit and pi-pulse calibration are useful prerequisites for all three
coherence measurements.

In [ ]:
exp.connect()

rabi_result = exp.obtain_rabi_params(
    targets=Q0,
    time_range=np.arange(16, 321, 16),
    n_shots=512,
    plot=True,
)
pi_result = exp.calibrate_pi_pulse(targets=Q0, n_shots=512, plot=True)

## 3. Define time grids on the hardware sampling grid

In [ ]:
t1_time_range = exp.util.discretize_time_range(
    np.geomspace(100, 100e3, 41),
    sampling_period=2,
)
t2_time_range = exp.util.discretize_time_range(
    np.geomspace(100, 50e3, 41),
    sampling_period=2,
)
ramsey_time_range = exp.util.discretize_time_range(
    np.geomspace(50, 20e3, 41),
    sampling_period=2,
)

## 4. Run T1 and T2 echo

In [ ]:
t1_result = exp.t1_experiment(
    targets=Q0,
    time_range=t1_time_range,
    n_shots=512,
    plot=True,
    xaxis_type="log",
)

t2_result = exp.t2_experiment(
    targets=Q0,
    time_range=t2_time_range,
    n_cpmg=1,
    n_shots=512,
    plot=True,
    xaxis_type="log",
)

## 5. Run Ramsey to refine the qubit frequency

Use a small detuning to extract the residual frequency error from the
oscillation envelope.

In [ ]:
ramsey_result = exp.ramsey_experiment(
    targets=Q0,
    time_range=ramsey_time_range,
    detuning=0.001,
    n_shots=512,
    plot=True,
)

## 6. Optional spectator-state Ramsey

If a neighboring qubit is configured, compare the Ramsey trace with the
spectator held in `|0>` and `|1>`.

In [ ]:
if spectator is not None:
    ramsey_spectator_0 = exp.ramsey_experiment(
        targets=Q0,
        time_range=ramsey_time_range,
        detuning=0.001,
        spectator_state="0",
        n_shots=512,
        plot=True,
    )
    ramsey_spectator_1 = exp.ramsey_experiment(
        targets=Q0,
        time_range=ramsey_time_range,
        detuning=0.001,
        spectator_state="1",
        n_shots=512,
        plot=True,
    )

## 7. Optionally persist the updated qubit frequency

In [ ]:
# exp.calib_note.save()
# exp.reload()